# Variational Autoencoders (VAEs)

Variational Autoencoders (VAEs) are a type of generative model that learn to encode data into a latent space and then decode from that space to reconstruct the data. VAEs are a probabilistic extension of traditional autoencoders and are designed to learn the underlying distribution of the data [1].

## Key Concepts

1. **Latent Variables**: VAEs introduce a set of latent variables $\mathbf{z}$ that represent the underlying structure of the data $\mathbf{x}$.
2. **Encoder (Inference Model)**: Maps input data $\mathbf{x}$ to a distribution over the latent variables $q_\phi(\mathbf{z}|\mathbf{x})$.
3. **Decoder (Generative Model)**: Maps latent variables $\mathbf{z}$ back to the data space, generating reconstructed data $p_\theta(\mathbf{x}|\mathbf{z})$.
4. **Prior Distribution**: The latent variables $\mathbf{z}$ are assumed to follow a prior distribution, typically a standard normal distribution $p(\mathbf{z}) = \mathcal{N}(\mathbf{z}; 0, \mathbf{I})$.

## Objective Function

The objective of a VAE is to maximize the Evidence Lower Bound (ELBO) on the marginal likelihood of the data [2]:

$$ \log p_\theta(\mathbf{x}) \geq \mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})} \left[ \log p_\theta(\mathbf{x}|\mathbf{z}) \right] - \text{KL}(q_\phi(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z})) $$

This ELBO consists of two terms:

1. **Reconstruction Term**: $\mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})} \left[ \log p_\theta(\mathbf{x}|\mathbf{z}) \right]$
2. **KL Divergence Term**: $\text{KL}(q_\phi(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z}))$.

### Reconstruction Term

The reconstruction term ensures that the decoded data is similar to the input data. This term can be viewed as a negative reconstruction error. In practice, for continuous data, it is often assumed that:

$$ p_\theta(\mathbf{x}|\mathbf{z}) = \mathcal{N}(\mathbf{x}; \mathbf{\mu}_\theta(\mathbf{z}), \sigma^2 \mathbf{I}) $$

The reconstruction term then becomes:

$$ \mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})} \left[ \log \mathcal{N}(\mathbf{x}; \mathbf{\mu}_\theta(\mathbf{z}), \sigma^2 \mathbf{I}) \right] $$

### KL Divergence Term

The KL divergence term regularizes the encoder to produce a latent distribution $q_\phi(\mathbf{z}|\mathbf{x})$ that is close to the prior $p(\mathbf{z})$ [3]:

$$ \text{KL}(q_\phi(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z})) = \int q_\phi(\mathbf{z}|\mathbf{x}) \log \frac{q_\phi(\mathbf{z}|\mathbf{x})}{p(\mathbf{z})} d\mathbf{z} $$

For a Gaussian prior and posterior, this term can be computed analytically.

## Variational Inference

To optimize the ELBO, we use variational inference. The encoder outputs the parameters of the posterior distribution $q_\phi(\mathbf{z}|\mathbf{x})$, typically the mean $\mathbf{\mu}_\phi(\mathbf{x})$ and variance $\mathbf{\sigma}_\phi(\mathbf{x})$ of a Gaussian distribution:

$$ q_\phi(\mathbf{z}|\mathbf{x}) = \mathcal{N}(\mathbf{z}; \mathbf{\mu}_\phi(\mathbf{x}), \text{diag}(\mathbf{\sigma}_\phi^2(\mathbf{x}))) $$

## Reparameterization Trick

To backpropagate through the stochastic sampling of $\mathbf{z}$, the reparameterization trick is used [1]. Instead of sampling $\mathbf{z}$ directly from $q_\phi(\mathbf{z}|\mathbf{x})$, we sample an auxiliary noise variable $\mathbf{\epsilon}$ from a standard normal distribution and transform it:

$$ \mathbf{z} = \mathbf{\mu}_\phi(\mathbf{x}) + \mathbf{\sigma}_\phi(\mathbf{x}) \odot \mathbf{\epsilon}, \quad \mathbf{\epsilon} \sim \mathcal{N}(0, \mathbf{I}) $$

This allows gradients to propagate through $\mathbf{\mu}_\phi(\mathbf{x})$ and $\mathbf{\sigma}_\phi(\mathbf{x})$ during training.

## Summary

The VAE learns to encode data $\mathbf{x}$ into a latent space $\mathbf{z}$ and decode $\mathbf{z}$ back to $\mathbf{x}$ while ensuring the latent space follows a standard normal distribution. The objective function combines a reconstruction error term and a regularization term to achieve this [4]:

$$ \mathcal{L}(\mathbf{x}) = \mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})} \left[ \log p_\theta(\mathbf{x}|\mathbf{z}) \right] - \text{KL}(q_\phi(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z})) $$

Optimizing this loss function results in a VAE that can generate new data points by sampling from the latent space and decoding them.

## References

1. Kingma, D. P., & Welling, M. (2013). Auto-Encoding Variational Bayes. arXiv preprint arXiv:1312.6114.
2. Doersch, C. (2016). Tutorial on Variational Autoencoders. arXiv preprint arXiv:1606.05908.
3. Rezende, D. J., Mohamed, S., & Wierstra, D. (2014). Stochastic Backpropagation and Approximate Inference in Deep Generative Models. arXiv preprint arXiv:1401.4082.
4. Goodfellow, I., Bengio, Y., & Courville, A. (2016). Deep Learning. MIT Press.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super(VAE, self).__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Mean and log variance layers
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )
    
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar

In [ ]:
def loss_function(recon_x, x, mu, logvar, beta=1.0):
    """
    VAE loss function combining reconstruction loss and KL divergence
    
    Args:
        recon_x: Reconstructed input
        x: Original input
        mu: Mean of the latent Gaussian distribution
        logvar: Log variance of the latent Gaussian distribution
        beta: Weight for the KL divergence term (beta-VAE)
    """
    # Reconstruction loss (Binary Cross Entropy)
    BCE = nn.functional.binary_cross_entropy(recon_x, x, reduction='sum')
    
    # KL divergence loss
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return BCE + beta * KLD, BCE, KLD

In [ ]:
# Load MNIST dataset
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: {batch_size}")
print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
def train_epoch(model, train_loader, optimizer, epoch, log_interval=100):
    model.train()
    train_loss = 0
    train_bce = 0
    train_kld = 0
    
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device).view(-1, 784)
        optimizer.zero_grad()
        
        recon_batch, mu, logvar = model(data)
        loss, bce, kld = loss_function(recon_batch, data, mu, logvar)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_bce += bce.item()
        train_kld += kld.item()
        
        if batch_idx % log_interval == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} '
                  f'({100. * batch_idx / len(train_loader):.0f}%)]\t'
                  f'Loss: {loss.item() / len(data):.6f}')
    
    avg_loss = train_loss / len(train_loader.dataset)
    avg_bce = train_bce / len(train_loader.dataset)
    avg_kld = train_kld / len(train_loader.dataset)
    
    print(f'====> Epoch: {epoch} Average loss: {avg_loss:.4f} '
          f'(BCE: {avg_bce:.4f}, KLD: {avg_kld:.4f})')
    
    return avg_loss, avg_bce, avg_kld


def test_epoch(model, test_loader):
    model.eval()
    test_loss = 0
    test_bce = 0
    test_kld = 0
    
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device).view(-1, 784)
            recon_batch, mu, logvar = model(data)
            loss, bce, kld = loss_function(recon_batch, data, mu, logvar)
            
            test_loss += loss.item()
            test_bce += bce.item()
            test_kld += kld.item()
    
    avg_loss = test_loss / len(test_loader.dataset)
    avg_bce = test_bce / len(test_loader.dataset)
    avg_kld = test_kld / len(test_loader.dataset)
    
    print(f'====> Test set loss: {avg_loss:.4f} '
          f'(BCE: {avg_bce:.4f}, KLD: {avg_kld:.4f})')
    
    return avg_loss, avg_bce, avg_kld

In [ ]:
def plot_reconstruction(model, data_loader, n_samples=8):
    """Plot original and reconstructed images"""
    model.eval()
    with torch.no_grad():
        for data, _ in data_loader:
            data = data.to(device)
            recon_data, _, _ = model(data.view(-1, 784))
            
            # Take only the first n_samples
            data = data[:n_samples]
            recon_data = recon_data[:n_samples]
            
            fig, axes = plt.subplots(2, n_samples, figsize=(12, 3))
            
            for i in range(n_samples):
                # Original images
                axes[0, i].imshow(data[i].cpu().squeeze(), cmap='gray')
                axes[0, i].set_title('Original')
                axes[0, i].axis('off')
                
                # Reconstructed images
                axes[1, i].imshow(recon_data[i].cpu().view(28, 28), cmap='gray')
                axes[1, i].set_title('Reconstructed')
                axes[1, i].axis('off')
            
            plt.tight_layout()
            plt.show()
            break


def plot_latent_space(model, data_loader, n_samples=1000):
    """Plot 2D latent space (only works for 2D latent space)"""
    if model.latent_dim != 2:
        print(f"Cannot plot latent space with {model.latent_dim} dimensions. Use 2D latent space.")
        return
    
    model.eval()
    latent_points = []
    labels = []
    
    with torch.no_grad():
        for data, label in data_loader:
            if len(latent_points) >= n_samples:
                break
            data = data.to(device).view(-1, 784)
            mu, _ = model.encode(data)
            latent_points.extend(mu.cpu().numpy())
            labels.extend(label.numpy())
    
    latent_points = np.array(latent_points[:n_samples])
    labels = np.array(labels[:n_samples])
    
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(latent_points[:, 0], latent_points[:, 1], c=labels, cmap='tab10')
    plt.colorbar(scatter)
    plt.title('Latent Space Representation')
    plt.xlabel('Latent Dimension 1')
    plt.ylabel('Latent Dimension 2')
    plt.show()


def generate_samples(model, n_samples=64):
    """Generate new samples from the latent space"""
    model.eval()
    with torch.no_grad():
        # Sample from standard normal distribution
        z = torch.randn(n_samples, model.latent_dim).to(device)
        samples = model.decode(z).cpu()
        
        # Plot the generated samples
        fig, axes = plt.subplots(8, 8, figsize=(10, 10))
        for i, ax in enumerate(axes.flat):
            if i < n_samples:
                ax.imshow(samples[i].view(28, 28), cmap='gray')
            ax.axis('off')
        
        plt.suptitle('Generated Samples from VAE')
        plt.tight_layout()
        plt.show()


def plot_training_curves(train_losses, test_losses, train_bce, test_bce, train_kld, test_kld):
    """Plot training and test curves"""
    epochs = range(1, len(train_losses) + 1)
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
    
    # Total loss
    ax1.plot(epochs, train_losses, 'b-', label='Train')
    ax1.plot(epochs, test_losses, 'r-', label='Test')
    ax1.set_title('Total Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    
    # Reconstruction loss (BCE)
    ax2.plot(epochs, train_bce, 'b-', label='Train')
    ax2.plot(epochs, test_bce, 'r-', label='Test')
    ax2.set_title('Reconstruction Loss (BCE)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('BCE Loss')
    ax2.legend()
    ax2.grid(True)
    
    # KL divergence
    ax3.plot(epochs, train_kld, 'b-', label='Train')
    ax3.plot(epochs, test_kld, 'r-', label='Test')
    ax3.set_title('KL Divergence')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('KLD Loss')
    ax3.legend()
    ax3.grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Initialize model and training parameters
latent_dim = 20  # Change to 2 for latent space visualization
hidden_dim = 400
learning_rate = 1e-3
epochs = 10

# Create model
model = VAE(input_dim=784, hidden_dim=hidden_dim, latent_dim=latent_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Latent dimension: {latent_dim}")
print(f"Hidden dimension: {hidden_dim}")
print(f"Learning rate: {learning_rate}")

In [ ]:
# Training loop
train_losses = []
test_losses = []
train_bce_losses = []
test_bce_losses = []
train_kld_losses = []
test_kld_losses = []

for epoch in range(1, epochs + 1):
    # Training
    train_loss, train_bce, train_kld = train_epoch(model, train_loader, optimizer, epoch)
    train_losses.append(train_loss)
    train_bce_losses.append(train_bce)
    train_kld_losses.append(train_kld)
    
    # Testing
    test_loss, test_bce, test_kld = test_epoch(model, test_loader)
    test_losses.append(test_loss)
    test_bce_losses.append(test_bce)
    test_kld_losses.append(test_kld)
    
    print()

print("Training completed!")

In [ ]:
def plot_reconstruction(model, data_loader, n_samples=8):
    """Plot original and reconstructed images"""
    model.eval()
    with torch.no_grad():
        for data, _ in data_loader:
            data = data.to(device)
            recon_data, _, _ = model(data.view(-1, 784))
            
            # Take only the first n_samples
            data = data[:n_samples]
            recon_data = recon_data[:n_samples]
            
            fig, axes = plt.subplots(2, n_samples, figsize=(12, 3))
            
            for i in range(n_samples):
                # Original images
                axes[0, i].imshow(data[i].cpu().squeeze(), cmap='gray')
                axes[0, i].set_title('Original')
                axes[0, i].axis('off')
                
                # Reconstructed images
                axes[1, i].imshow(recon_data[i].cpu().view(28, 28), cmap='gray')
                axes[1, i].set_title('Reconstructed')
                axes[1, i].axis('off')
            
            plt.tight_layout()
            plt.show()
            break


def plot_latent_space(model, data_loader, n_samples=1000):
    """Plot 2D latent space (only works for 2D latent space)"""
    if model.latent_dim != 2:
        print(f"Cannot plot latent space with {model.latent_dim} dimensions. Use 2D latent space.")
        return
    
    model.eval()
    latent_points = []
    labels = []
    
    with torch.no_grad():
        for data, label in data_loader:
            if len(latent_points) >= n_samples:
                break
            data = data.to(device).view(-1, 784)
            mu, _ = model.encode(data)
            latent_points.extend(mu.cpu().numpy())
            labels.extend(label.numpy())
    
    latent_points = np.array(latent_points[:n_samples])
    labels = np.array(labels[:n_samples])
    
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(latent_points[:, 0], latent_points[:, 1], c=labels, cmap='tab10')
    plt.colorbar(scatter)
    plt.title('Latent Space Representation')
    plt.xlabel('Latent Dimension 1')
    plt.ylabel('Latent Dimension 2')
    plt.show()


def generate_samples(model, n_samples=64):
    """Generate new samples from the latent space"""
    model.eval()
    with torch.no_grad():
        # Sample from standard normal distribution
        z = torch.randn(n_samples, model.latent_dim).to(device)
        samples = model.decode(z).cpu()
        
        # Plot the generated samples
        fig, axes = plt.subplots(8, 8, figsize=(10, 10))
        for i, ax in enumerate(axes.flat):
            if i < n_samples:
                ax.imshow(samples[i].view(28, 28), cmap='gray')
            ax.axis('off')
        
        plt.suptitle('Generated Samples from VAE')
        plt.tight_layout()
        plt.show()


def plot_training_curves(train_losses, test_losses, train_bce, test_bce, train_kld, test_kld):
    """Plot training and test curves"""
    epochs = range(1, len(train_losses) + 1)
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
    
    # Total loss
    ax1.plot(epochs, train_losses, 'b-', label='Train')
    ax1.plot(epochs, test_losses, 'r-', label='Test')
    ax1.set_title('Total Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    
    # Reconstruction loss (BCE)
    ax2.plot(epochs, train_bce, 'b-', label='Train')
    ax2.plot(epochs, test_bce, 'r-', label='Test')
    ax2.set_title('Reconstruction Loss (BCE)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('BCE Loss')
    ax2.legend()
    ax2.grid(True)
    
    # KL divergence
    ax3.plot(epochs, train_kld, 'b-', label='Train')
    ax3.plot(epochs, test_kld, 'r-', label='Test')
    ax3.set_title('KL Divergence')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('KLD Loss')
    ax3.legend()
    ax3.grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot training curves
plot_training_curves(train_losses, test_losses, train_bce_losses, test_bce_losses, train_kld_losses, test_kld_losses)

In [ ]:
# Show reconstructions
plot_reconstruction(model, test_loader, n_samples=8)

In [ ]:
# Generate new samples
generate_samples(model, n_samples=64)

In [ ]:
# Optional: Visualize latent space (only works with 2D latent space)
# Uncomment the lines below if you set latent_dim = 2

# model_2d = VAE(input_dim=784, hidden_dim=400, latent_dim=2).to(device)
# optimizer_2d = optim.Adam(model_2d.parameters(), lr=1e-3)

# # Train for a few epochs
# for epoch in range(1, 6):
#     train_epoch(model_2d, train_loader, optimizer_2d, epoch, log_interval=200)

# # Plot latent space
# plot_latent_space(model_2d, test_loader, n_samples=1000)